# KenProbe v0 training notebook

This notebook trains **KenProbe**, a compact citation-first research model. It learns when to search, how to emit tool calls, how to read source blocks, and how to answer with citations.

Run sections in order. Each section says whether to run or skip it.

**Real run:** 1000 steps / 5000 synthetic research rows / 2048 sequence length.

**Smoke test:** 80 steps / 500 synthetic research rows.

---

## Section 0 — GPU check

**Run.** You should see a Tesla T4. If not, reconnect to a GPU runtime before continuing.

In [ ]:
!nvidia-smi

---

## Section 1 — Install dependencies

**Run once per fresh runtime.**

If NumPy is upgraded and Unsloth complains, restart the kernel once, then continue from Section 2. Do not rerun this unless a package is missing.

In [ ]:
%%capture
!pip install --upgrade --force-reinstall --no-cache-dir unsloth unsloth_zoo
!pip install -U datasets trl accelerate bitsandbytes sentencepiece protobuf huggingface_hub pandas safetensors

---

## Section 2 — Imports and runtime check

**Run after every restart.**

This restores common imports. Later sections also include critical imports so the notebook is more resistant to VS Code/Colab state issues.

In [ ]:
import os, json, random, shutil
from datetime import datetime, timezone

import torch
from datasets import Dataset
from huggingface_hub import login, HfApi

print('Torch:', torch.__version__)
print('CUDA available:', torch.cuda.is_available())
if torch.cuda.is_available():
    print('GPU:', torch.cuda.get_device_name(0))

try:
    import numpy as np
    print('NumPy:', np.__version__)
except Exception as e:
    print('NumPy check failed:', repr(e))

---

## Section 3 — Hugging Face login

**Run before uploading adapters or GGUF.**

Uncomment `login()` and paste a write token. If you only want to train and test inside Colab, you can skip this until upload time.

In [ ]:
# from huggingface_hub import login
# login()

---

## Section 4 — Training config

**Run before model loading.**

Keep the defaults for real training. For a smoke test, set `MAX_STEPS = 80` and `TRAIN_HOTPOT_SAMPLES = 500`. The variable name is kept for compatibility, but the current stable dataset path is synthetic/local, not HotpotQA.

In [ ]:
BASE_MODEL = 'unsloth/Qwen3.5-4B'
# Fallback only if needed:
# BASE_MODEL = 'unsloth/Qwen3-4B-Instruct-2507'

PROJECT_NAME = 'kenprobe'
MODEL_SLUG = 'qwen35-4b'
HF_ADAPTER_REPO = 'amaansyed27/kenprobe-adapters'
HF_GGUF_REPO = 'amaansyed27/kenprobe-gguf'

MAX_SEQ_LENGTH = 2048
MAX_STEPS = 1000
TRAIN_HOTPOT_SAMPLES = 5000
EVAL_SIZE = 0.05
SEED = 3407
LOAD_IN_4BIT = False
LOAD_IN_16BIT = True

RUN_NAME = f'{PROJECT_NAME}-{MODEL_SLUG}-lora-step{MAX_STEPS}'
print('Run:', RUN_NAME)
print('Base model:', BASE_MODEL)
print('Rows:', TRAIN_HOTPOT_SAMPLES)
print('Steps:', MAX_STEPS)

---

## Section 5 — Load model and attach LoRA

**Run once after config.**

This loads Qwen and attaches trainable LoRA adapters.

In [ ]:
from unsloth import FastLanguageModel

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name=BASE_MODEL,
    max_seq_length=MAX_SEQ_LENGTH,
    load_in_4bit=LOAD_IN_4BIT,
    load_in_16bit=LOAD_IN_16BIT,
    fast_inference=False,
    full_finetuning=False,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    target_modules=['q_proj', 'k_proj', 'v_proj', 'o_proj', 'gate_proj', 'up_proj', 'down_proj'],
    lora_alpha=16,
    lora_dropout=0,
    bias='none',
    use_gradient_checkpointing='unsloth',
    random_state=SEED,
    max_seq_length=MAX_SEQ_LENGTH,
)
print('Model and LoRA adapter loaded.')

---

## Section 6 — Behavior prompt and formatting helpers

**Run before datasets.**

Defines the KenProbe behavior contract and text-only ChatML formatting.

In [ ]:
import json

SYSTEM_PROMPT = '''You are KenProbe, a citation-first research chatbot.

Rules:
1. For factual, current, benchmark, product, legal, medical, financial, or research questions, request search/tool use before answering.
2. Use structured tool calls in this exact XML-like wrapper:
   <tool_call>{\"name\":\"web_search\",\"arguments\":{\"query\":\"...\"}}</tool_call>
3. After sources are provided, answer only from the sources.
4. Cite source IDs inline as [S1], [S2].
5. If sources disagree, explain the disagreement.
6. If the evidence is insufficient, say "I do not have enough evidence" and explain what is missing.
7. Be concise, accurate, and avoid unsupported claims.
'''

text_tokenizer = getattr(tokenizer, 'tokenizer', tokenizer)

def to_chatml(messages, add_generation_prompt=False):
    parts = []
    for m in messages:
        parts.append(f"<|im_start|>{m['role']}\n{m['content']}<|im_end|>")
    if add_generation_prompt:
        parts.append('<|im_start|>assistant\n')
    return '\n'.join(parts)

def chat_text(messages):
    return to_chatml(messages, add_generation_prompt=False)

def make_tool_call(query):
    payload = {'name': 'web_search', 'arguments': {'query': query}}
    return '<tool_call>' + json.dumps(payload, ensure_ascii=False) + '</tool_call>'

def make_source_block(sources):
    return '\n\n'.join([f"[{s['id']}] {s['title']}\nURL: {s.get('url', 'local-dataset')}\n{s['text']}" for s in sources])

print('Behavior helpers ready.')

---

## Section 7 — Seed examples

**Run before dataset generation.**

These small examples teach the exact pattern: question → tool call → source block → cited answer.

In [ ]:
seed_examples = [
    {
        'user': 'Is a local 4B model enough to beat GPT-class models at research?',
        'query': 'small language models search tools research assistant accuracy',
        'sources': [
            {'id':'S1','title':'Research-agent design note','url':'local-seed://research-agent','text':'Small models can perform well in narrow workflows when paired with retrieval, tools, verification, and citations. They are weaker than frontier models as raw general reasoning systems.'},
            {'id':'S2','title':'RAG reliability note','url':'local-seed://rag','text':'Retrieval-augmented systems reduce hallucination risk when final answers are constrained to retrieved evidence and uncertainty is stated when evidence is insufficient.'},
        ],
        'answer': 'A 4B model is unlikely to beat frontier models as a raw general model. It can still be useful as a research system if it searches, verifies, and answers only from retrieved evidence [S1][S2].'
    },
    {
        'user': 'Should I answer current benchmark questions from memory?',
        'query': 'latest AI benchmark leaderboard current model scores',
        'sources': [{'id':'S1','title':'Benchmark freshness note','url':'local-seed://benchmarks','text':'Current model benchmark results change frequently. Answers about latest scores should be verified against current leaderboards or official model pages.'}],
        'answer': 'No. Current benchmark questions should trigger search first because leaderboard positions and model scores can change quickly [S1].'
    },
]

def seed_to_rows(examples):
    out = []
    for ex in examples:
        messages = [
            {'role':'system','content':SYSTEM_PROMPT},
            {'role':'user','content':ex['user']},
            {'role':'assistant','content':make_tool_call(ex['query'])},
            {'role':'tool','content':make_source_block(ex['sources'])},
            {'role':'assistant','content':ex['answer']},
        ]
        out.append({'text': chat_text(messages), 'source':'seed_tool_use'})
    return out

rows = seed_to_rows(seed_examples)
print('Seed rows:', len(rows))
print(rows[0]['text'][:1000])

---

## Section 8 — Build local research training rows

**Run after Section 7.**

This uses a stable local synthetic dataset instead of external Hugging Face datasets. This avoids Colab/HF dataset resolver errors while still teaching the protocol we need: search call → source block → cited answer → uncertainty.

In [ ]:
import random

assert 'rows' in globals(), 'Run Section 7 first so rows exists.'
assert 'TRAIN_HOTPOT_SAMPLES' in globals(), 'Run Section 4 first so row count exists.'
assert 'SYSTEM_PROMPT' in globals(), 'Run Section 6 first so SYSTEM_PROMPT exists.'

def build_local_research_rows(n=5000):
    topics = [
        {
            'topic': 'small local models for research',
            'question': 'Can small local models beat frontier models at research tasks?',
            'answer': 'Small local models are not proven to beat frontier models generally. They can become useful in narrow research workflows when paired with retrieval, tool use, verification, and citations.',
            'sources': [
                {'id':'S1','title':'Local model workflow note','url':'local://small-models','text':'Small local models can work well in narrow workflows when they are given retrieval, tool use, verification, and strict citation rules. They are weaker as general raw reasoning systems.'},
                {'id':'S2','title':'Research reliability note','url':'local://research-reliability','text':'Research assistants should avoid unsupported claims, cite evidence, and state uncertainty when evidence is insufficient.'},
            ],
        },
        {
            'topic': 'benchmark freshness',
            'question': 'Should benchmark questions be answered from memory?',
            'answer': 'No. Benchmark answers should be checked against current sources because leaderboards, model versions, and scores can change.',
            'sources': [{'id':'S1','title':'Benchmark freshness policy','url':'local://benchmarks','text':'Model benchmark results and leaderboard positions change frequently. Current claims should be verified against up-to-date sources.'}],
        },
        {
            'topic': 'medical claims',
            'question': 'Can I answer medical claims without checking sources?',
            'answer': 'No. Medical claims need source checking and uncertainty handling because incorrect advice can cause harm.',
            'sources': [{'id':'S1','title':'Medical safety note','url':'local://medical-safety','text':'Medical guidance is high stakes. A research assistant should check reliable sources and avoid giving unsupported diagnosis or treatment claims.'}],
        },
        {
            'topic': 'product recommendations',
            'question': 'Should product recommendations use current search?',
            'answer': 'Yes. Product recommendations should use current search because pricing, availability, specifications, and reviews change.',
            'sources': [{'id':'S1','title':'Product research note','url':'local://products','text':'Product information can change quickly, including price, availability, specifications, and user reviews.'}],
        },
        {
            'topic': 'insufficient evidence',
            'question': 'Is Model X proven to beat Model Y?',
            'answer': 'I do not have enough evidence to say that Model X beats Model Y. The available sources do not provide independent benchmark results.',
            'sources': [
                {'id':'S1','title':'Model X card','url':'local://model-x','text':'Model X claims strong performance in its own model card, but the claims are self-reported.'},
                {'id':'S2','title':'Independent leaderboard','url':'local://leaderboard','text':'The independent leaderboard does not list Model X, so there is no independent comparison against Model Y.'},
            ],
        },
    ]

    out = []
    for _ in range(n):
        item = random.choice(topics)
        user = random.choice([
            item['question'],
            f"Research this accurately: {item['question']}",
            f"Give me a cited answer about {item['topic']}.",
            f"Be careful and verify: {item['question']}",
        ])
        query = f"{item['topic']} evidence sources verification"
        source_ids = ''.join([f"[{s['id']}]" for s in item['sources'][:2]])
        cited_answer = f"{item['answer']} Evidence: {source_ids}"
        messages = [
            {'role':'system','content':SYSTEM_PROMPT},
            {'role':'user','content':user},
            {'role':'assistant','content':make_tool_call(query)},
            {'role':'tool','content':make_source_block(item['sources'])},
            {'role':'assistant','content':cited_answer},
        ]
        out.append({'text': chat_text(messages), 'source':'local_synthetic_research'})
    return out

dataset_rows = build_local_research_rows(TRAIN_HOTPOT_SAMPLES)
rows.extend(dataset_rows)
print('Loaded local dataset rows:', len(dataset_rows))
print('Total rows:', len(rows))
print('Source type:', dataset_rows[0]['source'])
print(rows[-1]['text'][:1200])

---

## Section 9 — Create train/eval split

**Run after Section 8.**

In [ ]:
from datasets import Dataset
import random

assert 'rows' in globals() and len(rows) > 10, 'Run Sections 7 and 8 first.'
assert 'SEED' in globals(), 'Run Section 4 first so SEED exists.'
assert 'EVAL_SIZE' in globals(), 'Run Section 4 first so EVAL_SIZE exists.'

random.seed(SEED)
random.shuffle(rows)

dataset = Dataset.from_list(rows).shuffle(seed=SEED)
split = dataset.train_test_split(test_size=EVAL_SIZE, seed=SEED)
train_dataset = split['train']
eval_dataset = split['test']

print(train_dataset)
print('Eval:', eval_dataset)
print('Preview:')
print(train_dataset[0]['text'][:2000])

---

## Section 10 — Train LoRA adapter

**Main training cell.**

For the default real run this trains for 1000 steps. Do not interrupt unless it errors.

In [ ]:
from trl import SFTTrainer, SFTConfig

trainer = SFTTrainer(
    model=model,
    tokenizer=text_tokenizer,
    train_dataset=train_dataset,
    eval_dataset=eval_dataset,
    args=SFTConfig(
        dataset_text_field='text',
        max_seq_length=MAX_SEQ_LENGTH,
        per_device_train_batch_size=1,
        gradient_accumulation_steps=4,
        warmup_steps=10,
        max_steps=MAX_STEPS,
        learning_rate=2e-4,
        logging_steps=1,
        output_dir=f'outputs_{RUN_NAME}',
        optim='adamw_8bit',
        seed=SEED,
        dataset_num_proc=1,
        fp16=True,
        bf16=False,
        report_to='none',
    ),
)

train_result = trainer.train()
print(train_result)

---

## Section 11 — Save adapter, report, and zip bundle

**Run immediately after training.**

In [ ]:
import os, json, shutil
from datetime import datetime, timezone

RUN_NAME = f'{PROJECT_NAME}-{MODEL_SLUG}-lora-step{MAX_STEPS}'
ADAPTER_DIR = f'adapters/{RUN_NAME}'
REPORT_DIR = 'reports'
BUNDLE_DIR = f'runs/{RUN_NAME}'
os.makedirs(ADAPTER_DIR, exist_ok=True)
os.makedirs(REPORT_DIR, exist_ok=True)
os.makedirs(BUNDLE_DIR, exist_ok=True)

model.save_pretrained(ADAPTER_DIR)
text_tokenizer.save_pretrained(ADAPTER_DIR)

report = {
    'run_name': RUN_NAME,
    'created_at': datetime.now(timezone.utc).isoformat(),
    'base_model': BASE_MODEL,
    'max_steps': MAX_STEPS,
    'max_seq_length': MAX_SEQ_LENGTH,
    'train_rows': TRAIN_HOTPOT_SAMPLES,
    'eval_size': EVAL_SIZE,
    'load_in_4bit': LOAD_IN_4BIT,
    'load_in_16bit': LOAD_IN_16BIT,
    'adapter_dir': ADAPTER_DIR,
    'hf_adapter_repo': HF_ADAPTER_REPO,
}
REPORT_PATH = f'{REPORT_DIR}/{RUN_NAME}.json'
with open(REPORT_PATH, 'w', encoding='utf-8') as f:
    json.dump(report, f, indent=2)

shutil.copytree(ADAPTER_DIR, f'{BUNDLE_DIR}/adapter', dirs_exist_ok=True)
shutil.copy(REPORT_PATH, f'{BUNDLE_DIR}/report.json')
BUNDLE_ZIP = shutil.make_archive(BUNDLE_DIR, 'zip', BUNDLE_DIR)
print('Saved adapter:', ADAPTER_DIR)
print('Saved report:', REPORT_PATH)
print('Created bundle:', BUNDLE_ZIP)
!ls -lh {BUNDLE_ZIP}

---

## Section 12 — Upload adapter bundle to HF

**Run after Section 11 if logged in.**

In [ ]:
import os
from huggingface_hub import HfApi, whoami

print(whoami())
api = HfApi()
api.create_repo(repo_id=HF_ADAPTER_REPO, repo_type='model', private=True, exist_ok=True)
api.upload_file(
    path_or_fileobj=BUNDLE_ZIP,
    path_in_repo=os.path.basename(BUNDLE_ZIP),
    repo_id=HF_ADAPTER_REPO,
    repo_type='model',
)
print('Uploaded:', f'{HF_ADAPTER_REPO}/{os.path.basename(BUNDLE_ZIP)}')

---

## Section 13 — Inference helper

**Run before test prompts.**

In [ ]:
import torch
from unsloth import FastLanguageModel

FastLanguageModel.for_inference(model)

def generate_text(messages, max_new_tokens=512, temperature=0.2):
    prompt = to_chatml(messages, add_generation_prompt=True)
    inputs = text_tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            pad_token_id=text_tokenizer.eos_token_id,
            eos_token_id=text_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[-1]:]
    return text_tokenizer.decode(new_tokens, skip_special_tokens=True)

print('Inference helper ready.')

---

## Section 14 — Test 1: search/tool-call behavior

Expected output should contain `<tool_call>{...}</tool_call>`.

In [ ]:
import torch
from unsloth import FastLanguageModel

assert 'model' in globals(), 'Model missing. Section 5/10 state is gone.'
assert 'text_tokenizer' in globals(), 'text_tokenizer missing. Run Section 6 or Section 13 again.'
assert 'to_chatml' in globals(), 'to_chatml missing. Run Section 6 again.'
assert 'SYSTEM_PROMPT' in globals(), 'SYSTEM_PROMPT missing. Run Section 6 again.'

FastLanguageModel.for_inference(model)

def generate_text(messages, max_new_tokens=512, temperature=0.2):
    prompt = to_chatml(messages, add_generation_prompt=True)
    inputs = text_tokenizer(prompt, return_tensors='pt', add_special_tokens=False).to('cuda')
    with torch.no_grad():
        outputs = model.generate(
            **inputs,
            max_new_tokens=max_new_tokens,
            temperature=temperature,
            top_p=0.9,
            do_sample=True,
            pad_token_id=text_tokenizer.eos_token_id,
            eos_token_id=text_tokenizer.eos_token_id,
        )
    new_tokens = outputs[0][inputs['input_ids'].shape[-1]:]
    return text_tokenizer.decode(new_tokens, skip_special_tokens=True)

test_messages = [
    {'role':'system','content':SYSTEM_PROMPT},
    {'role':'user','content':'Compare Qwythos 9B with Gemini Flash for research tasks. Be accurate.'},
]
print(generate_text(test_messages, max_new_tokens=350, temperature=0.2))

---

## Section 15 — Test 2: source-grounded answer behavior

Expected behavior: answer from `[S1]` and `[S2]`, and do not claim Qwythos beats Gemini Flash.

In [ ]:
assert 'generate_text' in globals(), 'Run Section 14 first.'

source_block = '''[S1] Qwythos model card
URL: https://huggingface.co/example/qwythos
Qwythos is a 9B model based on Qwen and advertises long-context capability. Its benchmark claims are self-reported on the model card.

[S2] Public benchmark leaderboard
URL: https://example.com/leaderboard
The public leaderboard does not list Qwythos. Frontier cloud models have independent agent benchmark scores that are much higher than small local models.
'''

test_messages = [
    {'role':'system','content':SYSTEM_PROMPT},
    {'role':'user','content':'Is Qwythos proven to beat Gemini Flash?'},
    {'role':'tool','content':source_block},
]
print(generate_text(test_messages, max_new_tokens=400, temperature=0.2))

---

## Section 16 — GGUF export and upload, fixed path

**Run only after Sections 11 and 12 are done.**

This section avoids compiling `llama.cpp` in Colab. It uses Unsloth's already-installed converter and quantizer. If Unsloth's first GGUF export failed after merging with `Can not map tensor model.layers.32...`, start from 16A.

Run in order:

1. 16A check merged folder
2. 16B patch layer count
3. 16C find converter/tools
4. 16D convert F16 GGUF
5. 16E quantize Q4_K_M
6. 16F upload GGUF

### Section 16A — Check merged model folder

Run this first. It checks the folder created before the failed Unsloth GGUF conversion.

In [32]:
import os, glob, json, re

RUN_NAME = 'kenprobe-qwen35-4b-lora-step1000'
MERGED_DIR = f'models/{RUN_NAME}-Q4_K_M'

print('MERGED_DIR:', MERGED_DIR)
print('Exists:', os.path.exists(MERGED_DIR))

print('Files:')
!find {MERGED_DIR} -maxdepth 2 -type f | head -80

safetensors_files = glob.glob(f'{MERGED_DIR}/**/*.safetensors', recursive=True)
print('Safetensors:', safetensors_files)

assert os.path.exists(MERGED_DIR), 'Merged folder missing. Run adapter save first or retry Unsloth merge until the merged folder exists.'
assert safetensors_files, 'No safetensors found in merged folder.'

MERGED_DIR: models/kenprobe-qwen35-4b-lora-step1000-Q4_K_M
Exists: True
Files:
models/kenprobe-qwen35-4b-lora-step1000-Q4_K_M/model.safetensors-00001-of-00002.safetensors
models/kenprobe-qwen35-4b-lora-step1000-Q4_K_M/chat_template.jinja
models/kenprobe-qwen35-4b-lora-step1000-Q4_K_M/model.safetensors-00002-of-00002.safetensors
models/kenprobe-qwen35-4b-lora-step1000-Q4_K_M/model.safetensors.index.json
models/kenprobe-qwen35-4b-lora-step1000-Q4_K_M/config.json
models/kenprobe-qwen35-4b-lora-step1000-Q4_K_M/tokenizer.json
models/kenprobe-qwen35-4b-lora-step1000-Q4_K_M/generation_config.json
models/kenprobe-qwen35-4b-lora-step1000-Q4_K_M/tokenizer_config.json
Safetensors: ['models/kenprobe-qwen35-4b-lora-step1000-Q4_K_M/model.safetensors-00001-of-00002.safetensors', 'models/kenprobe-qwen35-4b-lora-step1000-Q4_K_M/model.safetensors-00002-of-00002.safetensors']


### Section 16B — Patch config layer count

Run this before conversion. It fixes the `model.layers.32` mapping error by matching `config.json` to the actual layer tensors.

In [43]:
import os, glob, json, shutil, gc
from safetensors.torch import load_file, save_file

RUN_NAME = "kenprobe-qwen35-4b-lora-step1000"
MERGED_DIR = f"models/{RUN_NAME}-Q4_K_M"
CLEAN_DIR = f"models/{RUN_NAME}-clean-gguf"

assert os.path.exists(MERGED_DIR), "Merged folder missing."

if os.path.exists(CLEAN_DIR):
    shutil.rmtree(CLEAN_DIR)

os.makedirs(CLEAN_DIR, exist_ok=True)

# Copy all non-safetensors files first
for path in glob.glob(f"{MERGED_DIR}/*"):
    name = os.path.basename(path)
    if not name.endswith(".safetensors"):
        if os.path.isdir(path):
            shutil.copytree(path, os.path.join(CLEAN_DIR, name), dirs_exist_ok=True)
        else:
            shutil.copy2(path, os.path.join(CLEAN_DIR, name))

removed = []

# Rewrite each safetensors shard without orphan layer 32 tensors
for src in glob.glob(f"{MERGED_DIR}/*.safetensors"):
    name = os.path.basename(src)
    dst = os.path.join(CLEAN_DIR, name)

    print("Processing:", name)
    tensors = load_file(src, device="cpu")

    clean_tensors = {}
    for k, v in tensors.items():
        if k.startswith("model.layers.32."):
            removed.append(k)
        else:
            clean_tensors[k] = v

    save_file(clean_tensors, dst)

    del tensors
    del clean_tensors
    gc.collect()

# Patch index file
index_path = os.path.join(CLEAN_DIR, "model.safetensors.index.json")
if os.path.exists(index_path):
    with open(index_path, "r", encoding="utf-8") as f:
        index = json.load(f)

    old_count = len(index.get("weight_map", {}))
    index["weight_map"] = {
        k: v for k, v in index.get("weight_map", {}).items()
        if not k.startswith("model.layers.32.")
    }
    new_count = len(index["weight_map"])

    with open(index_path, "w", encoding="utf-8") as f:
        json.dump(index, f, indent=2)

    print("Index weights:", old_count, "→", new_count)

# Force config back to valid 32 layers
config_path = os.path.join(CLEAN_DIR, "config.json")
with open(config_path, "r", encoding="utf-8") as f:
    config = json.load(f)

config["num_hidden_layers"] = 32
if isinstance(config.get("text_config"), dict):
    config["text_config"]["num_hidden_layers"] = 32

with open(config_path, "w", encoding="utf-8") as f:
    json.dump(config, f, indent=2)

print("Removed orphan tensors:", removed)
print("Removed count:", len(removed))
print("Clean folder:", CLEAN_DIR)

assert len(removed) > 0, "No layer 32 tensors removed. Check folder state."

Processing: model.safetensors-00001-of-00002.safetensors
Processing: model.safetensors-00002-of-00002.safetensors
Index weights: 738 → 738
Removed orphan tensors: []
Removed count: 0
Clean folder: models/kenprobe-qwen35-4b-lora-step1000-clean-gguf


AssertionError: No layer 32 tensors removed. Check folder state.

### Section 16C — Find Unsloth prebuilt llama.cpp tools

No compile. This just locates the converter and quantizer Unsloth already downloaded.

In [44]:
import os, glob

UNSLOTH_LLAMA = '/root/.unsloth/llama.cpp'
CONVERT = f'{UNSLOTH_LLAMA}/unsloth_convert_hf_to_gguf.py'

quant_candidates = (
    glob.glob(f'{UNSLOTH_LLAMA}/**/llama-quantize', recursive=True)
    + glob.glob(f'{UNSLOTH_LLAMA}/**/quantize', recursive=True)
)

print('Unsloth llama.cpp exists:', os.path.exists(UNSLOTH_LLAMA))
print('Converter:', CONVERT, os.path.exists(CONVERT))
print('Quantizer candidates:', quant_candidates[:10])

assert os.path.exists(CONVERT), 'Unsloth converter not found. Run model.save_pretrained_gguf once until it installs llama.cpp, then stop/retry here.'
assert quant_candidates, 'No quantizer found under /root/.unsloth/llama.cpp.'

Unsloth llama.cpp exists: True
Converter: /root/.unsloth/llama.cpp/unsloth_convert_hf_to_gguf.py True
Quantizer candidates: ['/root/.unsloth/llama.cpp/llama-quantize']


### Section 16D — Convert merged HF folder to F16 GGUF

This uses the prebuilt Unsloth converter. It should not compile anything.

In [46]:
import os, subprocess, sys, shutil

RUN_NAME = "kenprobe-qwen35-4b-lora-step1000"
MERGED_DIR = f"models/{RUN_NAME}-Q4_K_M"
F16_GGUF = f"/content/{RUN_NAME}.F16.gguf"

LLAMA_SRC = "/content/llama.cpp-src"

assert os.path.exists(MERGED_DIR), "Merged model folder missing."

if os.path.exists(LLAMA_SRC):
    shutil.rmtree(LLAMA_SRC)

subprocess.run(
    ["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", LLAMA_SRC],
    check=True,
)

subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-r", f"{LLAMA_SRC}/requirements.txt"],
    check=True,
)

CONVERT = f"{LLAMA_SRC}/convert_hf_to_gguf.py"
assert os.path.exists(CONVERT), f"Missing converter: {CONVERT}"

if os.path.exists(F16_GGUF):
    os.remove(F16_GGUF)

cmd = [
    sys.executable,
    CONVERT,
    MERGED_DIR,
    "--outfile", F16_GGUF,
    "--outtype", "f16",
]

print("Running:")
print(" ".join(cmd))

result = subprocess.run(cmd, text=True, capture_output=True)

print("\n--- STDOUT LAST 5000 CHARS ---")
print(result.stdout[-5000:])

print("\n--- STDERR LAST 5000 CHARS ---")
print(result.stderr[-5000:])

if result.returncode != 0:
    raise RuntimeError(f"Fresh llama.cpp GGUF conversion failed with code {result.returncode}")

print("F16 GGUF created:", F16_GGUF)
!ls -lh {F16_GGUF}

Running:
/usr/bin/python3 /content/llama.cpp-src/convert_hf_to_gguf.py models/kenprobe-qwen35-4b-lora-step1000-Q4_K_M --outfile /content/kenprobe-qwen35-4b-lora-step1000.F16.gguf --outtype f16

--- STDOUT LAST 5000 CHARS ---


--- STDERR LAST 5000 CHARS ---
= {2560, 9216}
INFO:hf-to-gguf:blk.24.ffn_down.weight,               torch.bfloat16 --> F16, shape = {9216, 2560}
INFO:hf-to-gguf:blk.24.ffn_gate.weight,               torch.bfloat16 --> F16, shape = {2560, 9216}
INFO:hf-to-gguf:blk.24.ffn_up.weight,                 torch.bfloat16 --> F16, shape = {2560, 9216}
INFO:hf-to-gguf:blk.25.ffn_down.weight,               torch.bfloat16 --> F16, shape = {9216, 2560}
INFO:hf-to-gguf:blk.25.ffn_gate.weight,               torch.bfloat16 --> F16, shape = {2560, 9216}
INFO:hf-to-gguf:blk.25.ffn_up.weight,                 torch.bfloat16 --> F16, shape = {2560, 9216}
INFO:hf-to-gguf:blk.26.ffn_down.weight,               torch.bfloat16 --> F16, shape = {9216, 2560}
INFO:hf-to-gguf:blk.29.ffn_down.we

RuntimeError: Fresh llama.cpp GGUF conversion failed with code 1

### Section 16E — Quantize to Q4_K_M

Run after 16D creates the F16 GGUF.

In [ ]:
import os, glob, subprocess

RUN_NAME = 'kenprobe-qwen35-4b-lora-step1000'
F16_GGUF = f'/content/{RUN_NAME}.F16.gguf'
Q4_GGUF = f'/content/{RUN_NAME}.Q4_K_M.gguf'

quant_candidates = (
    glob.glob('/root/.unsloth/llama.cpp/**/llama-quantize', recursive=True)
    + glob.glob('/root/.unsloth/llama.cpp/**/quantize', recursive=True)
)
print('Quantizer candidates:', quant_candidates[:10])

assert os.path.exists(F16_GGUF), 'F16 GGUF missing. Run Section 16D first.'
assert quant_candidates, 'No quantizer found.'

QUANTIZE = quant_candidates[0]
subprocess.run([QUANTIZE, F16_GGUF, Q4_GGUF, 'Q4_K_M'], check=True)

print('Q4 GGUF created:', Q4_GGUF)
!ls -lh {Q4_GGUF}

### Section 16F — Upload Q4 GGUF to Hugging Face

Run after 16E. If this errors with 401, rerun Section 3 login, then run this cell again.

In [ ]:
import os
from huggingface_hub import HfApi, whoami

HF_GGUF_REPO = 'amaansyed27/kenprobe-gguf'
RUN_NAME = 'kenprobe-qwen35-4b-lora-step1000'
Q4_GGUF = f'/content/{RUN_NAME}.Q4_K_M.gguf'

assert os.path.exists(Q4_GGUF), 'Q4 GGUF not found. Run Section 16E first.'

print(whoami())
api = HfApi()
api.create_repo(repo_id=HF_GGUF_REPO, repo_type='model', private=True, exist_ok=True)
api.upload_file(
    path_or_fileobj=Q4_GGUF,
    path_in_repo=os.path.basename(Q4_GGUF),
    repo_id=HF_GGUF_REPO,
    repo_type='model',
)
print('Uploaded:', f'{HF_GGUF_REPO}/{os.path.basename(Q4_GGUF)}')

---

## Section 17 — Local download and website test

**Only read this. Run these in local PowerShell after adapter/GGUF uploads finish.**

Download adapter:

```powershell
cd C:\Users\Amaan\Downloads\ken-research
hf auth login
hf download amaansyed27/kenprobe-adapters kenprobe-qwen35-4b-lora-step1000.zip --local-dir runs
```

Download GGUF:

```powershell
hf download amaansyed27/kenprobe-gguf --local-dir models\kenprobe-gguf
dir .\models\kenprobe-gguf
```

Create Ollama model after replacing the GGUF file name:

```powershell
@'
FROM ./models/kenprobe-gguf/kenprobe-qwen35-4b-lora-step1000.Q4_K_M.gguf

SYSTEM """
You are KenProbe, a citation-first research chatbot.
For factual/current/research questions, request search/tool use before answering.
Use this exact tool-call format:
<tool_call>{"name":"web_search","arguments":{"query":"..."}}</tool_call>
After sources are provided, answer only from sources and cite [S1], [S2].
If evidence is insufficient, say so.
"""
'@ | Set-Content .\Modelfile.kenprobe

ollama create kenprobe:latest -f .\Modelfile.kenprobe
```

Run website with Tavily:

```powershell
cd C:\Users\Amaan\Downloads\ken-research\web
$env:BASELINE_MODEL="qwen3.5:4b"
$env:KENPROBE_MODEL="kenprobe:latest"
$env:SEARCH_PROVIDER="tavily"
$env:TAVILY_API_KEY="your_tavily_key_here"
npm run dev
```

Open `http://localhost:5177`.

In [47]:
import os, shutil, glob

RUN_NAME = "kenprobe-qwen35-4b-lora-step1000"
MERGED_16_DIR = f"models/{RUN_NAME}-merged-16bit"

if os.path.exists(MERGED_16_DIR):
    shutil.rmtree(MERGED_16_DIR)

# Save a clean merged HF model, not GGUF.
model.save_pretrained_merged(
    MERGED_16_DIR,
    text_tokenizer,
    save_method="merged_16bit",
)

print("Fresh merged model:", MERGED_16_DIR)
!find {MERGED_16_DIR} -maxdepth 1 -type f -print

Unsloth: Restored added_tokens_decoder metadata in models/kenprobe-qwen35-4b-lora-step1000-merged-16bit/tokenizer_config.json.


Found HuggingFace hub cache directory: /root/.cache/huggingface/hub


Fetching 1 files:   0%|          | 0/1 [00:00<?, ?it/s]

Checking cache directory for required files...


Unsloth: Copying 2 files from cache to `models/kenprobe-qwen35-4b-lora-step1000-merged-16bit`: 100%|██████████| 2/2 [02:35<00:00, 77.53s/it]


Successfully copied all 2 files from cache to `models/kenprobe-qwen35-4b-lora-step1000-merged-16bit`
Checking cache directory for required files...
Cache check failed: tokenizer.model not found in local cache.
Not all required files found in cache. Will proceed with downloading.


Unsloth: Merging weights into 16bit: 100%|██████████| 2/2 [02:18<00:00, 69.15s/it]


Unsloth: Merge process complete. Saved to `/content/models/kenprobe-qwen35-4b-lora-step1000-merged-16bit`
Fresh merged model: models/kenprobe-qwen35-4b-lora-step1000-merged-16bit
models/kenprobe-qwen35-4b-lora-step1000-merged-16bit/model.safetensors-00001-of-00002.safetensors
models/kenprobe-qwen35-4b-lora-step1000-merged-16bit/chat_template.jinja
models/kenprobe-qwen35-4b-lora-step1000-merged-16bit/model.safetensors-00002-of-00002.safetensors
models/kenprobe-qwen35-4b-lora-step1000-merged-16bit/model.safetensors.index.json
models/kenprobe-qwen35-4b-lora-step1000-merged-16bit/config.json
models/kenprobe-qwen35-4b-lora-step1000-merged-16bit/tokenizer.json
models/kenprobe-qwen35-4b-lora-step1000-merged-16bit/generation_config.json
models/kenprobe-qwen35-4b-lora-step1000-merged-16bit/tokenizer_config.json


In [48]:
import glob, re
from safetensors import safe_open

RUN_NAME = "kenprobe-qwen35-4b-lora-step1000"
MERGED_16_DIR = f"models/{RUN_NAME}-merged-16bit"

bad = []

for path in glob.glob(f"{MERGED_16_DIR}/*.safetensors"):
    with safe_open(path, framework="pt", device="cpu") as f:
        for key in f.keys():
            if "model.layers.32." in key:
                bad.append(key)

print("Bad layer 32 keys:", bad[:20])
print("Bad count:", len(bad))

assert len(bad) == 0, "Fresh merge still has invalid layer 32 tensors."

Bad layer 32 keys: []
Bad count: 0


In [49]:
import os, subprocess, sys, shutil

RUN_NAME = "kenprobe-qwen35-4b-lora-step1000"
MERGED_16_DIR = f"models/{RUN_NAME}-merged-16bit"
F16_GGUF = f"/content/{RUN_NAME}.F16.gguf"

LLAMA_SRC = "/content/llama.cpp-src"

if not os.path.exists(LLAMA_SRC):
    subprocess.run(
        ["git", "clone", "--depth", "1", "https://github.com/ggml-org/llama.cpp", LLAMA_SRC],
        check=True,
    )
    subprocess.run(
        [sys.executable, "-m", "pip", "install", "-q", "-r", f"{LLAMA_SRC}/requirements.txt"],
        check=True,
    )

CONVERT = f"{LLAMA_SRC}/convert_hf_to_gguf.py"

if os.path.exists(F16_GGUF):
    os.remove(F16_GGUF)

cmd = [
    sys.executable,
    CONVERT,
    MERGED_16_DIR,
    "--outfile", F16_GGUF,
    "--outtype", "f16",
]

print("Running:")
print(" ".join(cmd))

result = subprocess.run(cmd, text=True, capture_output=True)

print("\n--- STDERR LAST 5000 CHARS ---")
print(result.stderr[-5000:])

if result.returncode != 0:
    raise RuntimeError(f"Fresh merged conversion failed with code {result.returncode}")

print("F16 GGUF created:", F16_GGUF)
!ls -lh {F16_GGUF}

Running:
/usr/bin/python3 /content/llama.cpp-src/convert_hf_to_gguf.py models/kenprobe-qwen35-4b-lora-step1000-merged-16bit --outfile /content/kenprobe-qwen35-4b-lora-step1000.F16.gguf --outtype f16

--- STDERR LAST 5000 CHARS ---
bfloat16 --> F16, shape = {2560, 32}
INFO:hf-to-gguf:blk.8.ssm_beta.weight,                torch.bfloat16 --> F16, shape = {2560, 32}
INFO:hf-to-gguf:blk.8.attn_qkv.weight,                torch.bfloat16 --> F16, shape = {2560, 8192}
INFO:hf-to-gguf:blk.8.attn_gate.weight,               torch.bfloat16 --> F16, shape = {2560, 4096}
INFO:hf-to-gguf:blk.8.ssm_norm.weight,                torch.float32 --> F32, shape = {128}
INFO:hf-to-gguf:blk.8.ssm_out.weight,                 torch.bfloat16 --> F16, shape = {4096, 2560}
INFO:hf-to-gguf:blk.8.post_attention_norm.weight,     torch.bfloat16 --> F32, shape = {2560}
INFO:hf-to-gguf:blk.9.attn_norm.weight,               torch.bfloat16 --> F32, shape = {2560}
INFO:hf-to-gguf:blk.9.ssm_a,                          torch.f

RuntimeError: Fresh merged conversion failed with code 1

In [50]:
import os, json

RUN_NAME = "kenprobe-qwen35-4b-lora-step1000"
MERGED_16_DIR = f"models/{RUN_NAME}-merged-16bit"

TOKENIZER_CONFIG = f"{MERGED_16_DIR}/tokenizer_config.json"
assert os.path.exists(TOKENIZER_CONFIG), "tokenizer_config.json missing."

with open(TOKENIZER_CONFIG, "r", encoding="utf-8") as f:
    tok_cfg = json.load(f)

print("Old tokenizer_class:", tok_cfg.get("tokenizer_class"))

# Fix broken Unsloth tokenizer class
tok_cfg["tokenizer_class"] = "Qwen2TokenizerFast"

# Remove fields that can confuse AutoTokenizer / converter
for bad_key in [
    "auto_map",
    "processor_class",
]:
    if bad_key in tok_cfg:
        print("Removing:", bad_key)
        tok_cfg.pop(bad_key, None)

with open(TOKENIZER_CONFIG, "w", encoding="utf-8") as f:
    json.dump(tok_cfg, f, indent=2)

print("New tokenizer_class:", tok_cfg.get("tokenizer_class"))
print("Patched:", TOKENIZER_CONFIG)

Old tokenizer_class: TokenizersBackend
Removing: processor_class
New tokenizer_class: Qwen2TokenizerFast
Patched: models/kenprobe-qwen35-4b-lora-step1000-merged-16bit/tokenizer_config.json


In [51]:
import os, subprocess, sys

RUN_NAME = "kenprobe-qwen35-4b-lora-step1000"
MERGED_16_DIR = f"models/{RUN_NAME}-merged-16bit"
F16_GGUF = f"/content/{RUN_NAME}.F16.gguf"
LLAMA_SRC = "/content/llama.cpp-src"
CONVERT = f"{LLAMA_SRC}/convert_hf_to_gguf.py"

assert os.path.exists(CONVERT), f"Missing converter: {CONVERT}"

if os.path.exists(F16_GGUF):
    os.remove(F16_GGUF)

cmd = [
    sys.executable,
    CONVERT,
    MERGED_16_DIR,
    "--outfile", F16_GGUF,
    "--outtype", "f16",
]

print("Running:")
print(" ".join(cmd))

result = subprocess.run(cmd, text=True, capture_output=True)

print("\n--- STDERR LAST 5000 CHARS ---")
print(result.stderr[-5000:])

if result.returncode != 0:
    raise RuntimeError(f"Fresh merged conversion failed with code {result.returncode}")

print("F16 GGUF created:", F16_GGUF)
!ls -lh {F16_GGUF}

Running:
/usr/bin/python3 /content/llama.cpp-src/convert_hf_to_gguf.py models/kenprobe-qwen35-4b-lora-step1000-merged-16bit --outfile /content/kenprobe-qwen35-4b-lora-step1000.F16.gguf --outtype f16

--- STDERR LAST 5000 CHARS ---
 [02:46<00:34, 69.0Mbyte/s]
Writing:  73%|███████▎  | 6.29G/8.65G [02:46<00:31, 75.6Mbyte/s]
Writing:  73%|███████▎  | 6.34G/8.65G [02:47<00:27, 84.5Mbyte/s]
Writing:  73%|███████▎  | 6.36G/8.65G [02:47<00:25, 91.0Mbyte/s]
Writing:  74%|███████▎  | 6.38G/8.65G [02:47<00:25, 91.1Mbyte/s]
Writing:  74%|███████▍  | 6.40G/8.65G [02:47<00:22, 102Mbyte/s] 
Writing:  74%|███████▍  | 6.45G/8.65G [02:51<01:38, 22.3Mbyte/s]
Writing:  75%|███████▌  | 6.49G/8.65G [02:52<01:05, 32.8Mbyte/s]
Writing:  75%|███████▌  | 6.51G/8.65G [02:52<00:56, 38.1Mbyte/s]
Writing:  76%|███████▌  | 6.53G/8.65G [02:52<00:47, 44.3Mbyte/s]
Writing:  76%|███████▌  | 6.58G/8.65G [02:53<00:37, 55.8Mbyte/s]
Writing:  76%|███████▌  | 6.60G/8.65G [02:53<00:32, 63.6Mbyte/s]
Writing:  76%|███████▋  | 

In [52]:
import os, glob, subprocess

RUN_NAME = "kenprobe-qwen35-4b-lora-step1000"
F16_GGUF = f"/content/{RUN_NAME}.F16.gguf"
Q4_GGUF = f"/content/{RUN_NAME}.Q4_K_M.gguf"

quant_candidates = (
    glob.glob("/root/.unsloth/llama.cpp/**/llama-quantize", recursive=True)
    + glob.glob("/root/.unsloth/llama.cpp/**/quantize", recursive=True)
)

assert os.path.exists(F16_GGUF), "F16 GGUF missing."
assert quant_candidates, "No quantizer found."

QUANTIZE = quant_candidates[0]

if os.path.exists(Q4_GGUF):
    os.remove(Q4_GGUF)

subprocess.run([QUANTIZE, F16_GGUF, Q4_GGUF, "Q4_K_M"], check=True)

print("Q4 GGUF created:", Q4_GGUF)
!ls -lh {Q4_GGUF}

Q4 GGUF created: /content/kenprobe-qwen35-4b-lora-step1000.Q4_K_M.gguf
-rw-r--r-- 1 root root 2.6G Jun 30 13:08 /content/kenprobe-qwen35-4b-lora-step1000.Q4_K_M.gguf


In [53]:
import os
from huggingface_hub import HfApi, whoami

HF_GGUF_REPO = "amaansyed27/kenprobe-gguf"
RUN_NAME = "kenprobe-qwen35-4b-lora-step1000"
Q4_GGUF = f"/content/{RUN_NAME}.Q4_K_M.gguf"

assert os.path.exists(Q4_GGUF), "Q4 GGUF not found."

print(whoami())

api = HfApi()
api.create_repo(repo_id=HF_GGUF_REPO, repo_type="model", private=True, exist_ok=True)

api.upload_file(
    path_or_fileobj=Q4_GGUF,
    path_in_repo=os.path.basename(Q4_GGUF),
    repo_id=HF_GGUF_REPO,
    repo_type="model",
)

print("Uploaded:", f"{HF_GGUF_REPO}/{os.path.basename(Q4_GGUF)}")

{'type': 'user', 'id': '6937e2229f6bc8b7ec9bc35f', 'name': 'amaansyed27', 'fullname': 'Amaan Syed', 'canPay': False, 'billingMode': 'prepaid', 'periodEnd': 1782864000, 'isPro': False, 'avatarUrl': 'https://cdn-avatars.huggingface.co/v1/production/uploads/noauth/DZwgUypmq8N8yIhDRdZyT.png', 'orgs': [], 'auth': {'type': 'oauth', 'expiresAt': '2026-07-30T10:04:51.000Z'}}


Processing Files (0 / 0)      : |          |  0.00B /  0.00B            

New Data Upload               : |          |  0.00B /  0.00B            

  ...lora-step1000.Q4_K_M.gguf:   1%|          | 14.0MB / 2.78GB            

Uploaded: amaansyed27/kenprobe-gguf/kenprobe-qwen35-4b-lora-step1000.Q4_K_M.gguf
